#Configuration

In [0]:
STORAGE_ACCOUNT  = "cmsmedicaredatastorage"
CONTAINER        = "cms-medicare-raw"
SOURCE_PATH      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/readmissions/"
CHECKPOINT_PATH  = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/bronze_readmissions/"
TARGET_TABLE     = "cms_medicare_databricks_pipeline.bronze.readmissions_raw"
CATALOG          = "cms_medicare_databricks_pipeline"

#Auto Loader

In [0]:
#Auto Loader
df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
    .load(SOURCE_PATH)
)

print("Auto Loader stream defined")
print(f"Source: {SOURCE_PATH}")

#Add Metadata Columns

In [0]:
from pyspark.sql import functions as F
# Rename columns + add metadata since Delta Lake doesn't allow spaces or special characters in column names.
df_with_metadata = df_raw.select(
    F.col("Facility Name").alias("facility_name"),
    F.col("Facility ID").alias("facility_id"),
    F.col("State").alias("state"),
    F.col("Measure Name").alias("measure_name"),
    F.col("Number of Discharges").alias("number_of_discharges"),
    F.col("Footnote").alias("footnote"),
    F.col("Excess Readmission Ratio").alias("excess_readmission_ratio"),
    F.col("Predicted Readmission Rate").alias("predicted_readmission_rate"),
    F.col("Expected Readmission Rate").alias("expected_readmission_rate"),
    F.col("Number of Readmissions").alias("number_of_readmissions"),
    F.col("Start Date").alias("start_date"),
    F.col("End Date").alias("end_date"),
    # Metadata columns
    F.current_timestamp().alias("ingestion_timestamp"),
    F.input_file_name().alias("source_file")
)

print("✓ Columns renamed to snake_case + metadata added")
print(f"  Total columns: {len(df_with_metadata.columns)}")

#Write

In [0]:
(
    df_with_metadata
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

print("Bronze readmissions table written")

#Verify

In [0]:
%sql

SELECT COUNT(*) AS total_rows
FROM cms_medicare_databricks_pipeline.bronze.readmissions_raw;

In [0]:
df_raw_count = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(SOURCE_PATH)
)

raw_count = df_raw_count.count()
bronze_count = spark.read.table(TARGET_TABLE).count()

print(f"Raw CSV rows:      {raw_count:,}")
print(f"Bronze table rows: {bronze_count:,}")
print(f"Match: {raw_count == bronze_count} ✓" if raw_count == bronze_count 
      else f"MISMATCH: {raw_count - bronze_count:,} rows missing")

##Preview

In [0]:
%sql

SELECT *
FROM cms_medicare_databricks_pipeline.bronze.readmissions_raw
LIMIT 5